In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: "%.4f" % x)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

df = pd.read_csv("../../data/loan_default/loan_cleaned.csv")
print("Shape       :", df.shape)
print("Defaults    :", f"{df['is_default'].sum():,}")
print("Default rate:", round(df["is_default"].mean() * 100, 2), "%")\



Shape       : (2257159, 21)
Defaults    : 294,860
Default rate: 13.06 %


In [4]:
 # Encode Categorical Features
print("ENCODING CATEGORICAL FEATURES")
print()

grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
df["grade_encoded"] = df["grade"].map(grade_map)

home_map = {"MORTGAGE": 0, "RENT": 1, "OWN": 2, "ANY": 1, "OTHER": 1, "NONE": 1}
df["home_encoded"] = df["home_ownership"].map(home_map).fillna(1)

purpose_default_rates = df.groupby("purpose")["is_default"].mean()
df["purpose_risk_score"] = df["purpose"].map(purpose_default_rates)

print("Grade encoding (1=safest, 7=riskiest):")
for g, v in grade_map.items():
    rate = df[df["grade"] == g]["is_default"].mean() * 100
    print(f"  Grade {g} -> {v} | default rate: {rate:.2f}%")

print()
print("Purpose risk scores (mean default rate per purpose):")
print(purpose_default_rates.sort_values(ascending=False).head(8).round(4))

ENCODING CATEGORICAL FEATURES

Grade encoding (1=safest, 7=riskiest):
  Grade A -> 1 | default rate: 3.67%
  Grade B -> 2 | default rate: 8.81%
  Grade C -> 3 | default rate: 14.60%
  Grade D -> 4 | default rate: 20.67%
  Grade E -> 5 | default rate: 28.60%
  Grade F -> 6 | default rate: 36.75%
  Grade G -> 7 | default rate: 40.34%

Purpose risk scores (mean default rate per purpose):
purpose
educational          0.2090
small_business       0.2052
renewable_energy     0.1663
moving               0.1593
debt_consolidation   0.1412
medical              0.1367
other                0.1322
house                0.1294
Name: is_default, dtype: float64


In [5]:
# Financial Risk Features
print("ENGINEERING FINANCIAL RISK FEATURES")
print()

df["debt_to_income_ratio"] = df["dti"]
df["loan_to_income_ratio"] = df["loan_amnt"] / (df["annual_inc"] + 1)
df["installment_to_income"] = df["installment"] / (df["annual_inc"] / 12 + 1)
df["revolving_burden"] = df["revol_bal"] / (df["annual_inc"] + 1)
df["credit_utilization"] = df["revol_util"] / 100
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["fico_normalized"] = (df["fico_avg"] - df["fico_avg"].min()) / (df["fico_avg"].max() - df["fico_avg"].min())
df["int_rate_normalized"] = (df["int_rate"] - df["int_rate"].min()) / (df["int_rate"].max() - df["int_rate"].min())
df["composite_risk_score"] = (
    df["int_rate_normalized"] * 0.35 +
    df["grade_encoded"] / 7 * 0.30 +
    (1 - df["fico_normalized"]) * 0.25 +
    (df["dti"] / df["dti"].quantile(0.99).clip(1)).clip(0, 1) * 0.10
)
df["high_dti"] = (df["dti"] > 30).astype(int)
df["high_int_rate"] = (df["int_rate"] > 20).astype(int)
df["low_fico"] = (df["fico_range_low"] < 660).astype(int)
df["risky_grade"] = (df["grade_encoded"] >= 5).astype(int)

print("Features created:")
print("  loan_to_income_ratio   : loan amount relative to annual income")
print("  installment_to_income  : monthly payment relative to monthly income")
print("  revolving_burden       : revolving balance relative to income")
print("  credit_utilization     : revolving utilization as decimal")
print("  fico_avg               : average of fico low and high")
print("  composite_risk_score   : weighted combination of key risk signals")
print("  high_dti               : 1 if DTI > 30")
print("  high_int_rate          : 1 if interest rate > 20%")
print("  low_fico               : 1 if FICO < 660")
print("  risky_grade            : 1 if grade E, F, or G")
print()

binary_features = ["high_dti", "high_int_rate", "low_fico", "risky_grade"]
overall = df["is_default"].mean() * 100
print("Default rates for binary risk flags:")
for feat in binary_features:
    rate = df[df[feat] == 1]["is_default"].mean() * 100
    lift = rate / overall
    print(f"  {feat:<22}: {rate:.2f}% default rate  (lift: {lift:.2f}x)")


ENGINEERING FINANCIAL RISK FEATURES

Features created:
  loan_to_income_ratio   : loan amount relative to annual income
  installment_to_income  : monthly payment relative to monthly income
  revolving_burden       : revolving balance relative to income
  credit_utilization     : revolving utilization as decimal
  fico_avg               : average of fico low and high
  composite_risk_score   : weighted combination of key risk signals
  high_dti               : 1 if DTI > 30
  high_int_rate          : 1 if interest rate > 20%
  low_fico               : 1 if FICO < 660
  risky_grade            : 1 if grade E, F, or G

Default rates for binary risk flags:
  high_dti              : 16.73% default rate  (lift: 1.28x)
  high_int_rate         : 27.12% default rate  (lift: 2.08x)
  low_fico              : 30.80% default rate  (lift: 2.36x)
  risky_grade           : 31.15% default rate  (lift: 2.38x)


In [6]:
# Credit History Features
print("ENGINEERING CREDIT HISTORY FEATURES")
print()

df["delinq_per_year"] = df["delinq_2yrs"] / 2
df["has_delinquency"] = (df["delinq_2yrs"] > 0).astype(int)
df["has_public_record"] = (df["pub_rec"] > 0).astype(int)
df["has_bankruptcy"] = (df["pub_rec_bankruptcies"] > 0).astype(int)
df["credit_depth"] = df["total_acc"]
df["credit_breadth"] = df["open_acc"] / (df["total_acc"] + 1)
df["mort_acc_flag"] = (df["mort_acc"] > 0).astype(int).fillna(0)
df["emp_length_clean"] = df["emp_length"].fillna(df["emp_length"].median())
df["experienced_borrower"] = (df["emp_length_clean"] >= 5).astype(int)
df["multiple_derogatory"] = ((df["delinq_2yrs"] > 1) | (df["pub_rec"] > 1)).astype(int)

print("Features created:")
print("  delinq_per_year       : delinquencies normalized per year")
print("  has_delinquency       : 1 if any delinquency in 2 years")
print("  has_public_record     : 1 if any public record")
print("  has_bankruptcy        : 1 if any bankruptcy")
print("  credit_breadth        : ratio of open to total accounts")
print("  experienced_borrower  : 1 if employed 5+ years")
print("  multiple_derogatory   : 1 if multiple negative marks")
print()

credit_features = ["has_delinquency", "has_public_record", "has_bankruptcy", "multiple_derogatory"]
print("Default rates for credit history flags:")
for feat in credit_features:
    rate_yes = df[df[feat] == 1]["is_default"].mean() * 100
    rate_no = df[df[feat] == 0]["is_default"].mean() * 100
    print(f"  {feat:<25}: with={rate_yes:.2f}%  without={rate_no:.2f}%")

ENGINEERING CREDIT HISTORY FEATURES

Features created:
  delinq_per_year       : delinquencies normalized per year
  has_delinquency       : 1 if any delinquency in 2 years
  has_public_record     : 1 if any public record
  has_bankruptcy        : 1 if any bankruptcy
  credit_breadth        : ratio of open to total accounts
  experienced_borrower  : 1 if employed 5+ years
  multiple_derogatory   : 1 if multiple negative marks

Default rates for credit history flags:
  has_delinquency          : with=14.54%  without=12.73%
  has_public_record        : with=15.85%  without=12.54%
  has_bankruptcy           : with=15.34%  without=12.75%
  multiple_derogatory      : with=16.10%  without=12.79%


In [7]:
#  Feature Importance Ranking
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("FEATURE IMPORTANCE RANKING")
print()

encode_cols = ["grade", "home_ownership", "purpose"]
df_model = df.copy()
le = LabelEncoder()
for col in encode_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

drop_cols = ["is_default"]
feature_cols = [c for c in df_model.columns if c not in drop_cols]
feature_cols = [c for c in feature_cols if df_model[c].dtype in ["float64", "int64", "int32", "float32"]]

sample_df = df_model.sample(100000, random_state=42)
X = sample_df[feature_cols].fillna(0)
y = sample_df["is_default"]

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight="balanced", max_depth=8)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 20 most important features:")
for i, (feat, imp) in enumerate(importances.head(20).items(), 1):
    tag = "ENGINEERED" if feat not in ["loan_amnt", "funded_amnt", "int_rate", "installment",
                                         "annual_inc", "dti", "delinq_2yrs", "fico_range_low",
                                         "fico_range_high", "open_acc", "pub_rec", "revol_bal",
                                         "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
                                         "emp_length", "grade", "home_ownership", "purpose"] else "original"
    print(f"  {i:2}. {feat:<30} {imp:.6f}  [{tag}]")

FEATURE IMPORTANCE RANKING

Top 20 most important features:
   1. composite_risk_score           0.155912  [ENGINEERED]
   2. grade                          0.134500  [original]
   3. grade_encoded                  0.119661  [ENGINEERED]
   4. int_rate                       0.100741  [original]
   5. int_rate_normalized            0.095055  [ENGINEERED]
   6. risky_grade                    0.042236  [ENGINEERED]
   7. loan_to_income_ratio           0.028793  [ENGINEERED]
   8. installment_to_income          0.026912  [ENGINEERED]
   9. fico_normalized                0.022014  [ENGINEERED]
  10. fico_range_low                 0.020660  [original]
  11. fico_range_high                0.019401  [original]
  12. debt_to_income_ratio           0.018087  [ENGINEERED]
  13. dti                            0.017733  [original]
  14. annual_inc                     0.014941  [original]
  15. fico_avg                       0.013690  [ENGINEERED]
  16. revol_bal                      0.013312  [orig

In [10]:
# Saving Engineered Dataset
print("SAVING ENGINEERED DATASET")
print()

drop_for_model = ["grade", "home_ownership", "purpose", "emp_length"]
df_final = df.drop(columns=[c for c in drop_for_model if c in df.columns])
df_final = df_final.fillna(0)

df_final.to_csv("../../data/loan_default/loan_engineered.csv", index=False)

original_features = ["loan_amnt", "funded_amnt", "int_rate", "installment", "annual_inc",
                      "dti", "delinq_2yrs", "fico_range_low", "fico_range_high", "open_acc",
                      "pub_rec", "revol_bal", "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies"]
engineered_features = [c for c in df_final.columns if c not in original_features + ["is_default"]]

print(f"Original features    : {len(original_features)}")
print(f"Engineered features  : {len(engineered_features)}")
print(f"Total features       : {df_final.shape[1] - 1}")
print(f"Total rows           : {len(df_final):,}")
print(f"Default rate         : {df_final['is_default'].mean()*100:.2f}%")
print()
print("Engineered features added:")
for f in engineered_features:
    print(f"  {f}")
print()



SAVING ENGINEERED DATASET

Original features    : 16
Engineered features  : 26
Total features       : 42
Total rows           : 2,257,159
Default rate         : 13.06%

Engineered features added:
  grade_encoded
  home_encoded
  purpose_risk_score
  debt_to_income_ratio
  loan_to_income_ratio
  installment_to_income
  revolving_burden
  credit_utilization
  fico_avg
  fico_normalized
  int_rate_normalized
  composite_risk_score
  high_dti
  high_int_rate
  low_fico
  risky_grade
  delinq_per_year
  has_delinquency
  has_public_record
  has_bankruptcy
  credit_depth
  credit_breadth
  mort_acc_flag
  emp_length_clean
  experienced_borrower
  multiple_derogatory

